# Test the OpenAI API

Companion to `10_ollama_test.ipynb`: list models, test a simple response, send chat messages, and extract structured ministry data. Run from the project root.

Use your existing `.env` setting `OPENAI_API_KEY=...` (lowercase `openai_api_key` is also accepted). The key is loaded without printing it. Change `llm_model` below to choose another model available to your account.

Examples follow the [official OpenAI structured output documentation](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=responses) and use the [GPT-4.1 mini model](https://developers.openai.com/api/docs/models/gpt-4.1-mini).


## 0 Environment setup
Install project dependencies in this notebook's environment, then create the API client using the key from `.env`. Existing environment variables take precedence.

After installing dependencies, restart the kernel and run from the setup cell. The client requests gzip/deflate compression to avoid the saved Brotli `process() takes no keyword arguments` failure.


In [1]:
%pip install -e .


Obtaining file:///C:/Users/joema/projects/570-project
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for parish-ministry-benchmark (pyproject.toml): started
  Building editable for parish-ministry-benchmark (pyproject.toml): finished with status 'done'
  Created wheel for parish-ministry-benchmark: filename=parish_ministry_benchmark-0.1.0-0.editable-py3-none-any.whl size=2987 sha256=bb38a5ace274e0a4f6b5392a1a2feb7d68de80e1a7f1236b2ca6f1b2f934695b
  Stored in directory: C:\Users\joema\AppData\Local\Temp\pip-ephem-wheel-cache-4

In [2]:
import os

from dotenv import load_dotenv
from openai import OpenAI

_ = load_dotenv(".env", override=False)
api_key = os.environ.get("OPENAI_API_KEY") or os.environ.get("openai_api_key")
if not api_key or api_key == "your-key-here":
    raise ValueError("Set OPENAI_API_KEY in the project-root .env file before running this notebook.")

# Avoid the incompatible Brotli decoder in some Anaconda environments.
client = OpenAI(
    api_key=api_key,
    default_headers={"Accept-Encoding": "gzip, deflate"},
)
# llm_model = "gpt-4.1-mini"
llm_model = "gpt-4.1-nano"


## 1 List available models


In [3]:
for model in sorted(client.models.list(), key=lambda model: model.id):
    print(model.id)

print(f"Selected model: {llm_model}")


babbage-002
chat-latest
chatgpt-image-latest
davinci-002
gpt-3.5-turbo
gpt-3.5-turbo-0125
gpt-3.5-turbo-1106
gpt-3.5-turbo-16k
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
gpt-4
gpt-4-0613
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4.1
gpt-4.1-2025-04-14
gpt-4.1-mini
gpt-4.1-mini-2025-04-14
gpt-4.1-nano
gpt-4.1-nano-2025-04-14
gpt-4o
gpt-4o-2024-05-13
gpt-4o-2024-08-06
gpt-4o-2024-11-20
gpt-4o-mini
gpt-4o-mini-2024-07-18
gpt-4o-mini-search-preview
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-transcribe
gpt-4o-mini-transcribe-2025-03-20
gpt-4o-mini-transcribe-2025-12-15
gpt-4o-mini-tts
gpt-4o-mini-tts-2025-03-20
gpt-4o-mini-tts-2025-12-15
gpt-4o-search-preview
gpt-4o-search-preview-2025-03-11
gpt-4o-transcribe
gpt-4o-transcribe-diarize
gpt-5
gpt-5-2025-08-07
gpt-5-chat-latest
gpt-5-codex
gpt-5-mini
gpt-5-mini-2025-08-07
gpt-5-nano
gpt-5-nano-2025-08-07
gpt-5-pro
gpt-5-pro-2025-10-06
gpt-5-search-api
gpt-5-search-api-2025-10-14
gpt-5.1
gpt-5.1-2025-11-13
gpt-5.1-chat-latest
gpt-5.

## 2 Test a simple response


In [4]:
response = client.responses.create(
    model=llm_model,
    input="Reply with exactly: OpenAI is working",
    store=False,
)

print(response.output_text)


OpenAI is working


## 3 Test chat messages


In [5]:
response = client.responses.create(
    model=llm_model,
    input=[
        {"role": "user", "content": "Say hello and tell me what 2 + 2 is."},
    ],
    store=False,
)

print(response.output_text)


Hello! 2 + 2 is 4.


## 4 Extract ministries with structured output
Use the same sample and Pydantic result shape as the Ollama test. Each ministry has a name and one category. The SDK parses the response into `Result`.


In [6]:
from pydantic import BaseModel


class Ministry(BaseModel):
    name: str
    category: str


class Result(BaseModel):
    ministries: list[Ministry]


response = client.responses.parse(
    model=llm_model,
    input=[
        {"role": "system", "content": "Extract Catholic parish ministries and assign one category to each ministry."},
        {
            "role": "user",
            "content": """
A Catholic parish has an Altar Server ministry and
a St. Vincent de Paul Society.

Extract the ministries and categorize them.
""",
        },
    ],
    text_format=Result,
    temperature=0,
    store=False,
)

result = response.output_parsed
if result is None:
    raise RuntimeError("No structured result was returned; inspect response for a refusal or incomplete response.")

print(result)
print(result.model_dump_json(indent=2))


ministries=[Ministry(name='Altar Server', category='Liturgy and Worship'), Ministry(name='St. Vincent de Paul Society', category='Charitable Works')]
{
  "ministries": [
    {
      "name": "Altar Server",
      "category": "Liturgy and Worship"
    },
    {
      "name": "St. Vincent de Paul Society",
      "category": "Charitable Works"
    }
  ]
}
